In [ ]:
from datasets import DatasetDict, Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np
import os
import sys 
import sklearn
import torch 
import random
import pandas as pd

seed = 100
# torch.manual_seed(seed)
# torch.cuda.manual_seed(seed)
# torch.cuda.manual_seed_all(seed)

# random.seed(seed)
# np.random.seed(seed)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False

REPO_NAME = "A_Blockchain_Enabled_Framework_for_Misinformation_Monitoring"
REPO_URL = "https://github.com/SnowWoofer/A_Blockchain_Enabled_Framework_for_Misinformation_Monitoring.git"

if os.path.exists("/kaggle"):
    BASE_DIR = f"/kaggle/working/{REPO_NAME}"
    os.environ["DATA_DIR"] = f"/kaggle/working/{REPO_NAME}"
elif os.path.exists("/content"):    
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = "/content/drive/MyDrive/blockchain_misinformation_data"
    os.makedirs(os.environ["DATA_DIR"], exist_ok=True)
    BASE_DIR = f"/content/{REPO_NAME}"
else:
    BASE_DIR = "."

if not os.path.exists(BASE_DIR):
    !git clone {REPO_URL} {REPO_NAME}

os.chdir(BASE_DIR)
sys.path.append(os.path.join(BASE_DIR, "src"))

In [ ]:
dataset_dict =  load_dataset("roupenminassian/twitter-misinformation")
#defien the pre tained modle to be used 
model_path = "Davlan/afro-xlmr-large"

#load the models tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

#load model with binary classifictaions head
id2label = {0: "not misinfomration", 1: "misinformation"}
label2id = {"not misinformation":0, "misinformation":1}
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
# freeze all base modle paramets excpet opr lpoolingg layers(near clasisifictaion head)
for name, param in model.base_model.named_parameters():
    if "pooler" in name:
        param.requires_grad=True
    else:
        param.requires_grad=False

        

In [ ]:
# data preprocessing

#split dataset:

df_train, df_test = np.split()


#Define text preprocessing
def preprocess_function(examples):
    # return tokenized text with truncation
    return tokenizer(examples["text"], truncation=True)

#preprocess all datasets
tokenized_data = dataset_dict.map(preprocess_function, batched=True)

Map:   0%|          | 0/92394 [00:00<?, ? examples/s]

Map:   0%|          | 0/10267 [00:00<?, ? examples/s]

In [12]:
#create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("../data/raw/twitter_data_eng_raw.csv",encoding='utf-8')
#print(df.head())

train = df[df['set']=='train']
test = df[df['set']=='test']

print(train.head())
print(test.head())


   Unnamed: 0    set                                               text  \
0           0  train  Local Charlotte, NC news station WSOCTV is rep...   
1           1  train  The tsunami has started President Obama s Keny...   
2           2  train  The only reality show Donald Trump should have...   
3           3  train  No Food, No FEMA: Hurricane Michael’s Survivor...   
4           4  train  Here are some of the highlights of the Reuters...   

   label                                             source  
0      1  pic.twitter.com/GGuM2Ow3wk, @wsoctv), @wsoctv,...  
1      1                                  entire story: NYP  
2      1   Featured Image: Christopher Furlong/Getty Images  
3      0                          https://thebea.st/2CIXNoz  
4      0                             WASHINGTON (Reuters) -  
       Unnamed: 0   set                                               text  \
92394       92394  test  It's official....Cross High School will be use...   
92395       92395  te